# TMDb Movie Data Analysis: Exploratory Data Analysis & Business Intelligence

**Author:** Data Analyst Portfolio Project  
**Dataset:** `mymoviedb.csv` (The Movie Database - TMDb)  
**Tools:** Python, Pandas, Matplotlib, Seaborn, SQLite  

---

## 1. Project Objective
The objective of this project is to perform an end-to-end exploratory data analysis on a catalog of **9,827 popular movies** sourced from The Movie Database (TMDb). 

This analysis aims to answer six core business questions:
1. **Content Portfolio Composition:** Which movie genres dominate the catalog, and what is their penetration across film titles?
2. **Linguistic & Regional Coverage:** How is the catalog distributed across global languages, and where are the international content gaps?
3. **Historical Production Velocity:** How has annual movie release volume evolved over the decades, and what macroeconomic trends or shocks are observable?
4. **Audience Reception & Genre Quality:** Which genres achieve the highest average ratings when controlling for sample-size bias?
5. **Commercial Popularity vs. Critical Acclaim:** Does high commercial popularity reliably predict higher audience appreciation?
6. **Data Integrity & Pipeline Hygiene:** How can automated discretization bugs be avoided to safeguard unrated catalog inventory?

## 2. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization styles
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Resolve dataset path dynamically
data_path = '../data/mymoviedb.csv' if os.path.exists('../data/mymoviedb.csv') else 'data/mymoviedb.csv'
df_raw = pd.read_csv(data_path, lineterminator='\n')

print(f'Dataset successfully loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head(3)

## 3. Data Inspection & Cleaning Pipeline

Before performing analysis, we inspect data types, missing values, duplicates, and numerical boundaries.

In [ ]:
# Initial schema inspection
print('--- Dataset Info ---')
print(df_raw.info())

print('\n--- Missing Values by Column ---')
print(df_raw.isnull().sum())

print(f'\nExact Full-Row Duplicates: {df_raw.duplicated().sum()}')
print(f'Unique Movie Titles: {df_raw["Title"].nunique()}')
print(f'Duplicate Title Occurrences: {df_raw.duplicated(subset=["Title"]).sum()}')

### Duplicate Titles Investigation
Notice that there are 314 duplicate title rows. We inspect whether these represent duplicate entries or legitimate multi-year remakes.

In [ ]:
same_date_dups = df_raw.duplicated(subset=['Title', 'Release_Date']).sum()
print(f'Duplicate Titles with Identical Release Dates: {same_date_dups}')

# Sample of duplicate titles
dups_sample = df_raw[df_raw.duplicated(subset=['Title'], keep=False)].sort_values('Title')
dups_sample[['Release_Date', 'Title', 'Vote_Average', 'Vote_Count']].head(6)

> **Data Quality Finding:** There are **zero duplicate records** sharing both title and release date. All duplicate titles represent legitimate remakes or adaptations released across different years (e.g., *A Nightmare on Elm Street* in 1984 vs. 2010).

### Data Cleaning & Handling Zero-Vote Unrated Movies

In TMDb, when a film has received no user votes (`Vote_Count == 0`), the API returns a default `Vote_Average` of `0.0`.
- Treating these as true scores of zero distorts rating averages.
- Deleting them drops valid catalog titles from inventory.
- **Solution:** We explicitly classify them into `Rating_Status = 'Unrated'`, keeping them in the catalog while calculating clean averages on verified rated titles.

In [ ]:
df = df_raw.copy()

# 1. Assign surrogate primary key
df.insert(0, 'movie_id', range(1, len(df) + 1))

# 2. Convert Release_Date and extract temporal features
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')
df['Release_Year'] = df['Release_Date'].dt.year.astype(int)
df['Release_Month'] = df['Release_Date'].dt.month.astype(int)
df['Decade'] = (df['Release_Year'] // 10) * 10

# 3. Handle zero-vote unrated titles
df['Rating_Status'] = np.where(df['Vote_Count'] == 0, 'Unrated', 'Rated')
unrated_count = (df['Rating_Status'] == 'Unrated').sum()
print(f'Preserved Unrated Titles: {unrated_count} (out of {len(df)} total)')

# Clean Rating Average (excluding unrated zeros)
unadjusted_mean = df['Vote_Average'].mean()
clean_mean = df[df['Rating_Status'] == 'Rated']['Vote_Average'].mean()
print(f'Unadjusted Average Rating (including zeros): {unadjusted_mean:.2f}')
print(f'Clean Average Rating (rated titles only):    {clean_mean:.2f}')

### Genre Normalization & Multi-Label Handling
Films in the catalog are multi-genre comma-separated strings. To analyze genres without corrupting movie-level metrics, we create a normalized genre bridge table.

In [ ]:
records = []
for _, row in df.iterrows():
    m_id = row['movie_id']
    for g in str(row['Genre']).split(','):
        clean_g = g.strip()
        if clean_g:
            records.append({'movie_id': m_id, 'genre': clean_g})

df_genres = pd.DataFrame(records)
print(f'Total Normalized Genre Tags: {len(df_genres)}')
print(f'Unique Genre Categories:     {df_genres["genre"].nunique()}')
print(f'Average Genres per Film:     {len(df_genres) / len(df):.2f}')

## 4. Exploratory Data Analysis & Visualizations

### Chart 1: Top 10 Movie Genres (Title Penetration Rate %)
We calculate what percentage of unique catalog titles feature each genre tag ($N = 9,827$ titles).

In [ ]:
genre_counts = df_genres['genre'].value_counts().reset_index()
genre_counts.columns = ['genre', 'tag_count']
genre_counts['title_penetration_pct'] = (genre_counts['tag_count'] / len(df)) * 100.0

plt.figure(figsize=(10, 5.5))
sns.barplot(
    data=genre_counts.head(10),
    x='title_penetration_pct',
    y='genre',
    palette='Blues_r'
)
plt.title('Top 10 Genres by Title Penetration Rate (% of Catalog Titles)', fontsize=13, fontweight='bold')
plt.xlabel('Percentage of Catalog Films (%)')
plt.ylabel('Genre')
for i, row in genre_counts.head(10).iterrows():
    plt.text(row['title_penetration_pct'] + 0.5, i, f"{row['title_penetration_pct']:.1f}% ({row['tag_count']:,})", va='center', fontsize=10)
plt.xlim(0, 45)
plt.tight_layout()
plt.show()

### Chart 2: Historical Movie Release Velocity (1980–2022)
Analyzing annual movie release counts and isolating the 2020 pandemic dip and 2021 backlog recovery.

In [ ]:
modern_releases = df[df['Release_Year'].between(1980, 2022)]['Release_Year'].value_counts().sort_index().reset_index()
modern_releases.columns = ['Release_Year', 'Releases']

plt.figure(figsize=(11, 5))
plt.plot(modern_releases['Release_Year'], modern_releases['Releases'], marker='o', color='#2b5c8f', linewidth=2, markersize=5)
plt.axvline(x=2020, color='#e74c3c', linestyle='--', alpha=0.7, label='2020 Pandemic Disruption')
plt.axvline(x=2021, color='#27ae60', linestyle='--', alpha=0.7, label='2021 Backlog Peak (714 titles)')
plt.title('Annual Movie Releases (1980–2022): Growth, Disruption & Peak Rebound', fontsize=13, fontweight='bold')
plt.xlabel('Release Year')
plt.ylabel('Number of Films')
plt.legend()
plt.tight_layout()
plt.show()

### Chart 3: Global Language Distribution (Top 6 Non-English vs. English)

In [ ]:
lang_counts = df['Original_Language'].value_counts()
en_share = (lang_counts['en'] / len(df)) * 100.0
print(f'English Content Share: {en_share:.2f}% ({lang_counts["en"]} titles)')

top_international = lang_counts.drop('en').head(6).reset_index()
top_international.columns = ['Language_Code', 'Titles']
top_international['Share_Pct'] = (top_international['Titles'] / len(df)) * 100.0

plt.figure(figsize=(9, 4.5))
sns.barplot(data=top_international, x='Language_Code', y='Titles', palette='viridis')
plt.title('Top 6 International (Non-English) Languages by Film Volume', fontsize=13, fontweight='bold')
plt.xlabel('Language Code (ISO-639-1)')
plt.ylabel('Film Count')
for i, row in top_international.iterrows():
    plt.text(i, row['Titles'] + 15, f"{row['Titles']} ({row['Share_Pct']:.1f}%)", ha='center', fontsize=10)
plt.ylim(0, 750)
plt.tight_layout()
plt.show()

### Chart 4: Audience Rating Distribution (Rated Titles Only)

In [ ]:
rated_df = df[df['Rating_Status'] == 'Rated']

plt.figure(figsize=(10, 4.5))
sns.histplot(rated_df['Vote_Average'], bins=30, kde=True, color='#34495e')
plt.axvline(rated_df['Vote_Average'].mean(), color='#e74c3c', linestyle='-', label=f'Mean: {rated_df["Vote_Average"].mean():.2f}')
plt.axvline(rated_df['Vote_Average'].median(), color='#f39c12', linestyle='--', label=f'Median: {rated_df["Vote_Average"].median():.2f}')
plt.title('Audience Vote Average Distribution (Cleaned Rated Films)', fontsize=13, fontweight='bold')
plt.xlabel('Vote Average (0 to 10)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

### Chart 5: Commercial Popularity vs. Audience Acclaim
We examine whether higher commercial popularity scores correspond to higher audience ratings.

In [ ]:
sample_plot = rated_df.sample(2500, random_state=42)

plt.figure(figsize=(10, 5))
plt.scatter(sample_plot['Vote_Average'], sample_plot['Popularity'], alpha=0.35, color='#2980b9', edgecolors='none', s=25)
plt.yscale('log')
plt.title('Commercial Popularity vs. Audience Rating (Log Scale)', fontsize=13, fontweight='bold')
plt.xlabel('Audience Vote Average (0 to 10)')
plt.ylabel('Popularity Index (Log Scale)')
plt.tight_layout()
plt.show()

# Correlation test
corr = rated_df[['Popularity', 'Vote_Average']].corr(method='spearman').iloc[0, 1]
print(f'Spearman Rank Correlation (Popularity vs. Rating): {corr:.4f}')

## 5. Key Analytical Findings

1. **Core Genre Domination:** Drama (**38.10%** title penetration, 3,744 films) and Comedy (**30.84%**, 3,031 films) represent the foundational pillars of the catalog. The top 4 genres (Drama, Comedy, Action, Thriller) represent **46.33% of all genre tags**.
2. **Anglo-Centric Bias with Japanese Outperformance:** English represents **77.03% (7,570 titles)** of the catalog. Japanese (`ja`) is the second-largest individual language at **6.56% (645 titles)**, surpassing Spanish (**3.45%**, 339 titles) and French (**2.97%**, 292 titles).
3. **Production Volatility & Backlog Surge:** Catalog releases expanded by **+28.21% between 2015 and 2019** (390 to 500 titles), experienced a **-9.40% contraction in 2020** (453 titles) during global filming shutdowns, and rebounded with an all-time peak of **714 titles in 2021 (+57.62% YoY)**.
4. **Disconnect Between Popularity and Acclaim:** Spearman rank correlation between Popularity and Rating is weak (**rho = 0.127**), proving that high commercial buzz does not correlate with critical viewer appreciation.
5. **Data Pipeline Protection:** Exactly **100 titles (1.02%)** were identified as unrated (`Vote_Count == 0`). Preserving these titles in an `Unrated` tier prevents artificial rating deflation while maintaining accurate catalog volume metrics.

## 6. Business Recommendations

1. **Diversify Beyond Drama/Comedy Saturation:** Standard Drama and Comedy acquisitions yield diminishing differentiation. Platforms should invest in high-fandom verticals like **Animation (14.64%)** and **Sci-Fi (12.95%)**, which typically deliver higher merchandising and long-term viewer engagement.
2. **Capitalize on Latin American & Spanish Demand:** Spanish-language content (3.45%) is heavily under-indexed relative to the 500M+ global Spanish-speaking population, pointing to a high-leverage acquisition opportunity in Latin America and Spain.
3. **Balance New Releases with Evergreen Catalogs:** Due to production vulnerability (as seen in 2020's -9.4% drop), platforms must license evergreen catalogs (films released before 2015 comprise 57.67% of the inventory) to stabilize subscriber engagement during production pauses.
4. **Avoid Pure Popularity Recommendation Bias:** Because popularity weakly correlates with rating, recommendation systems should balance trending popularity with high-consensus ratings to ensure quality discovery.

## 7. Limitations & Conclusion

* **Dataset Scope:** This dataset contains movie catalog metadata from TMDb, not internal platform subscriber streaming logs.
* **Temporal Metric:** Trends reflect studio release dates (`Release_Date`), not streaming ingestion schedules (`date_added`).
* **Conclusion:** By applying disciplined data cleaning, multi-label normalization, and exploratory visualization, this analysis transformed raw movie records into actionable content strategy benchmarks.